# Black–Scholes Simulation + Neural-Network Trading Agent

**A self-contained notebook for Google Colab.**

This notebook does four things:

1. **Simulates** the price dynamics that underpin the **Black–Scholes (BS)** model
   (Geometric Brownian Motion) and implements the closed-form BS option pricer and
   its *Greeks*.
2. **Builds a neural network** that learns a **trading decision policy**
   (go long / stay flat / go short) from features engineered around the BS /
   volatility framework.
3. **Backtests** the resulting algorithm on **real historical market data**.
4. **Reports the results as charts** (equity curve, drawdown, Sharpe, signal
   distribution, etc.).

Everything is written in English and the relevant **theory is included inline**.

---

### Table of contents
1. [Setup & imports](#setup)
2. [Part I — Black–Scholes theory](#bs-theory)
3. [Part I — Geometric Brownian Motion simulation](#gbm)
4. [Part I — Black–Scholes pricing & Greeks](#bs-pricing)
5. [Part II — Neural-network trading: theory](#nn-theory)
6. [Part II — Real historical data](#data)
7. [Part II — Feature engineering (BS / volatility based)](#features)
8. [Part II — Building & training the network](#train)
9. [Part III — Backtesting the strategy](#backtest)
10. [Part III — Results & charts](#results)
11. [Conclusions & caveats](#conclusions)

> ⚠️ **Disclaimer.** This notebook is for **education and research** only. It is
> *not* financial advice. Backtested performance does not guarantee future
> results, and the model deliberately keeps things simple for clarity.


<a id="setup"></a>
## 1. Setup & imports

Run this cell first. In Colab, `numpy`, `pandas`, `matplotlib`, `scipy`,
`scikit-learn` and `tensorflow` are already installed; we only need to add
`yfinance` for downloading real market data.


In [ ]:
# Install the one dependency Colab may be missing.
# (Safe to re-run; it is a no-op if already installed.)
import sys, subprocess
try:
    import yfinance  # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)

# TensorFlow / Keras
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("NumPy   :", np.__version__)
print("Pandas  :", pd.__version__)
print("TF/Keras:", tf.__version__)


<a id="bs-theory"></a>
## 2. Part I — Black–Scholes theory

### 2.1 The model of the underlying

The Black–Scholes framework assumes the price of the underlying asset
$S_t$ follows a **Geometric Brownian Motion (GBM)**:

$$
dS_t = \mu\, S_t\, dt + \sigma\, S_t\, dW_t,
$$

where

- $\mu$ is the (real-world) **drift** / expected return,
- $\sigma$ is the **volatility**,
- $W_t$ is a standard **Brownian motion** (Wiener process).

Applying Itô's lemma to $\ln S_t$ gives the closed-form solution

$$
S_t = S_0 \, \exp\!\Big[\big(\mu - \tfrac12\sigma^2\big)t + \sigma W_t\Big],
$$

so **log-returns are normally distributed** and prices are **log-normal**.

### 2.2 The Black–Scholes PDE

Under a no-arbitrage argument with continuous **delta-hedging**, the value
$V(S,t)$ of any European derivative satisfies the **Black–Scholes PDE**:

$$
\frac{\partial V}{\partial t}
+ \tfrac12\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}
+ r S \frac{\partial V}{\partial S}
- r V = 0,
$$

where $r$ is the **risk-free rate**. Note the drift $\mu$ disappears: pricing is
done under the **risk-neutral measure** where the asset drifts at $r$.

### 2.3 Closed-form European option prices

For a European **call** ($C$) and **put** ($P$) with strike $K$ and time to
maturity $T$:

$$
d_1 = \frac{\ln(S/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}},
\qquad
d_2 = d_1 - \sigma\sqrt{T},
$$

$$
C = S\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2),
\qquad
P = K e^{-rT}\,\Phi(-d_2) - S\,\Phi(-d_1),
$$

where $\Phi$ is the standard normal CDF. Put–call parity holds:
$C - P = S - K e^{-rT}$.

### 2.4 The Greeks

The **Greeks** are sensitivities of the option value used for hedging and risk.
For a call:

| Greek | Meaning | Formula |
|-------|---------|---------|
| $\Delta$ | $\partial V/\partial S$ | $\Phi(d_1)$ |
| $\Gamma$ | $\partial^2 V/\partial S^2$ | $\dfrac{\phi(d_1)}{S\sigma\sqrt{T}}$ |
| $\mathcal{V}$ (Vega) | $\partial V/\partial \sigma$ | $S\,\phi(d_1)\sqrt{T}$ |
| $\Theta$ | $\partial V/\partial t$ | $-\dfrac{S\phi(d_1)\sigma}{2\sqrt{T}} - rKe^{-rT}\Phi(d_2)$ |
| $\rho$ | $\partial V/\partial r$ | $KTe^{-rT}\Phi(d_2)$ |

$\phi$ is the standard normal PDF. **Delta** is the key link to trading: it is the
number of units of the underlying needed to hedge the option, and — as we'll use
later — a natural, bounded way to translate a *directional forecast* into a
*position size*.


<a id="gbm"></a>
## 3. Part I — Simulating Geometric Brownian Motion

We now simulate GBM paths (the "world" the BS model assumes) and check that the
empirical distribution of log-returns matches the theory.


In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, seed=None):
    '''Simulate Geometric Brownian Motion paths.

    dS = mu*S*dt + sigma*S*dW  ->  exact log-Euler scheme.

    Returns
    -------
    t     : (n_steps+1,) time grid
    paths : (n_paths, n_steps+1) simulated price paths
    '''
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    # Brownian increments
    dW = rng.normal(0.0, np.sqrt(dt), size=(n_paths, n_steps))
    # Log-return increments (exact solution of GBM)
    incr = (mu - 0.5 * sigma**2) * dt + sigma * dW
    log_paths = np.concatenate(
        [np.zeros((n_paths, 1)), np.cumsum(incr, axis=1)], axis=1
    )
    paths = S0 * np.exp(log_paths)
    t = np.linspace(0.0, T, n_steps + 1)
    return t, paths


# Parameters
S0, mu, sigma, T = 100.0, 0.08, 0.20, 1.0
n_steps, n_paths = 252, 200

t, paths = simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, seed=SEED)
print("Simulated", paths.shape[0], "paths over", paths.shape[1], "time points.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# (a) A sample of simulated paths
for i in range(40):
    ax[0].plot(t, paths[i], lw=0.8, alpha=0.6)
ax[0].plot(t, S0 * np.exp(mu * t), color="black", lw=2.5, label=r"$E[S_t]=S_0e^{\mu t}$")
ax[0].set_title("Simulated GBM price paths")
ax[0].set_xlabel("Time (years)"); ax[0].set_ylabel("Price"); ax[0].legend()

# (b) Terminal log-returns vs the theoretical normal density
terminal_log_ret = np.log(paths[:, -1] / S0)
ax[1].hist(terminal_log_ret, bins=30, density=True, alpha=0.6, label="Simulated")
xs = np.linspace(terminal_log_ret.min(), terminal_log_ret.max(), 200)
theo = norm.pdf(xs, (mu - 0.5 * sigma**2) * T, sigma * np.sqrt(T))
ax[1].plot(xs, theo, "r-", lw=2, label="Theoretical N")
ax[1].set_title("Terminal log-returns vs Black–Scholes theory")
ax[1].set_xlabel(r"$\ln(S_T/S_0)$"); ax[1].set_ylabel("Density"); ax[1].legend()

plt.tight_layout(); plt.show()


<a id="bs-pricing"></a>
## 4. Part I — Black–Scholes pricing & Greeks

We implement the closed-form pricer and the Greeks, then visualise how the call
price and its Delta behave across the moneyness spectrum.


In [ ]:
def bs_price(S, K, T, r, sigma, option="call"):
    '''Black–Scholes price of a European option (vectorised).'''
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    T = np.maximum(T, 1e-12); sigma = np.maximum(sigma, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def bs_greeks(S, K, T, r, sigma, option="call"):
    '''Return a dict with Delta, Gamma, Vega, Theta, Rho.'''
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    T = np.maximum(T, 1e-12); sigma = np.maximum(sigma, 1e-12)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = norm.pdf(d1)
    if option == "call":
        delta = norm.cdf(d1)
        theta = (-S * pdf * sigma / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * norm.cdf(d2))
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:
        delta = norm.cdf(d1) - 1.0
        theta = (-S * pdf * sigma / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * norm.cdf(-d2))
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
    gamma = pdf / (S * sigma * np.sqrt(T))
    vega = S * pdf * np.sqrt(T)
    return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho": rho}


# Sanity check: put-call parity  C - P = S - K e^{-rT}
K, r = 100.0, 0.03
C = bs_price(100, K, 1.0, r, 0.2, "call")
P = bs_price(100, K, 1.0, r, 0.2, "put")
print(f"Call = {C:.4f}, Put = {P:.4f}")
print(f"C - P = {C - P:.4f}  vs  S - K e^-rT = {100 - K*np.exp(-r*1.0):.4f}")


In [ ]:
S_grid = np.linspace(60, 140, 200)
call_prices = bs_price(S_grid, K, T=0.5, r=r, sigma=0.2, option="call")
greeks = bs_greeks(S_grid, K, T=0.5, r=r, sigma=0.2, option="call")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(S_grid, call_prices, lw=2)
ax[0].axvline(K, color="gray", ls="--", label="Strike K")
ax[0].plot(S_grid, np.maximum(S_grid - K, 0), "k:", label="Payoff at maturity")
ax[0].set_title("Black–Scholes call price"); ax[0].set_xlabel("Spot S")
ax[0].set_ylabel("Option value"); ax[0].legend()

ax[1].plot(S_grid, greeks["delta"], lw=2, label=r"$\Delta$ (call)")
ax[1].plot(S_grid, greeks["gamma"] * 10, lw=2, label=r"$\Gamma \times 10$")
ax[1].axvline(K, color="gray", ls="--")
ax[1].set_title("Delta & Gamma vs spot"); ax[1].set_xlabel("Spot S")
ax[1].set_ylabel("Greek value"); ax[1].legend()
plt.tight_layout(); plt.show()


<a id="nn-theory"></a>
## 5. Part II — Neural-network trading: theory

### 5.1 Idea

Classical Black–Scholes assumes **constant, known volatility** and prices
derivatives; it does **not** tell you *which direction* the market will move.
Here we take the complementary view: we keep the **volatility/return machinery of
the BS world** and let a **neural network learn a directional decision** from data.

The pipeline is:

$$
\underbrace{\text{market data}}_{\text{prices}}
\;\rightarrow\;
\underbrace{\text{BS / volatility features}}_{\text{returns, }\sigma,\text{ z-scores, Greeks}}
\;\rightarrow\;
\underbrace{\text{neural network}}_{\text{classifier}}
\;\rightarrow\;
\underbrace{\text{position}}_{\text{long / flat / short}}
\;\rightarrow\;
\underbrace{\text{backtest}}_{\text{P\&L, Sharpe}}
$$

### 5.2 What the network predicts

We frame trading as a **3-class classification** of the **next day's return**:

- class **+1 (Long)**  if next-day return $> +\tau$,
- class **0  (Flat)**  if $|$next-day return$| \le \tau$,
- class **−1 (Short)** if next-day return $< -\tau$,

with a small **dead-band** $\tau$ (a fraction of daily volatility) so the model
is not forced to bet on noise. The network outputs class probabilities via a
**softmax**; the trading position is a **volatility-scaled, Delta-like mapping**
of those probabilities into $[-1, +1]$.

### 5.3 Why Black–Scholes features?

- **Realized volatility** $\sigma$ is *the* BS parameter and strongly drives
  risk-adjusted returns; we feed several horizons of it.
- **Standardized moves** $z = r_t / \sigma_t$ are exactly the argument of the
  normal distribution in the BS formula — natural, scale-free features.
- A synthetic **BS Delta** built from a rolling z-score gives a smooth, bounded
  "how far in/out of the money is momentum" signal.

### 5.4 Avoiding look-ahead bias

Financial ML is easy to get wrong. We are careful to:

- build every feature from **past** data only (rolling windows, then `shift`),
- split **chronologically** (train → validation → test, never shuffled),
- fit the scaler on the **training set only**,
- apply realistic **transaction costs** in the backtest.


<a id="data"></a>
## 6. Part II — Real historical data

We download real daily prices with `yfinance`. If the Colab runtime has no
internet access (or the download fails), we **fall back to a GBM-simulated
series** so the whole notebook still runs end-to-end.


In [ ]:
TICKER   = "SPY"          # try e.g. "AAPL", "MSFT", "^GSPC", "BTC-USD"
START    = "2010-01-01"
END      = "2024-12-31"

def load_prices(ticker, start, end):
    '''Return a DataFrame with a 'Close' column, real or simulated.'''
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df is not None and len(df) > 250:
            out = df[["Close"]].dropna().copy()
            out.attrs["source"] = f"real data ({ticker})"
            return out
        raise ValueError("empty download")
    except Exception as e:
        print(f"[warn] download failed ({e}); using GBM-simulated fallback series.")
        n = 252 * 12
        _, p = simulate_gbm(100.0, 0.07, 0.18, n / 252, n, 1, seed=SEED)
        idx = pd.bdate_range(start=start, periods=n + 1)
        out = pd.DataFrame({"Close": p[0]}, index=idx)
        out.attrs["source"] = "SIMULATED (offline fallback)"
        return out

prices = load_prices(TICKER, START, END)
SOURCE = prices.attrs.get("source", "unknown")
print("Data source:", SOURCE)
print("Rows:", len(prices), "| from", prices.index[0].date(), "to", prices.index[-1].date())
prices.tail()


In [ ]:
plt.figure()
plt.plot(prices.index, prices["Close"], lw=1.2)
plt.title(f"{TICKER} closing price  [{SOURCE}]")
plt.xlabel("Date"); plt.ylabel("Price"); plt.tight_layout(); plt.show()


<a id="features"></a>
## 7. Part II — Feature engineering (BS / volatility based)

Every feature below is computed from **past** information only. The realized
volatility is the annualized standard deviation of log-returns — the empirical
counterpart of the BS $\sigma$.


In [ ]:
def build_features(prices, vol_windows=(5, 10, 21, 63), mom_windows=(5, 10, 21, 63)):
    df = pd.DataFrame(index=prices.index)
    df["close"] = prices["Close"]
    df["log_ret"] = np.log(df["close"]).diff()

    # --- Realized (annualized) volatility over several horizons: the BS sigma ---
    for w in vol_windows:
        df[f"vol_{w}"] = df["log_ret"].rolling(w).std() * np.sqrt(252)

    # --- Momentum / trend features ---
    for w in mom_windows:
        df[f"mom_{w}"] = df["close"].pct_change(w)

    # --- Standardized daily move  z = r_t / sigma_t  (BS-normal argument) ---
    df["z_score"] = df["log_ret"] / (df["log_ret"].rolling(21).std() + 1e-9)

    # --- Distance from moving averages (moneyness-like) ---
    for w in (21, 63):
        ma = df["close"].rolling(w).mean()
        df[f"dist_ma_{w}"] = (df["close"] - ma) / ma

    # --- RSI(14): a bounded momentum oscillator ---
    delta = df["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    df["rsi_14"] = 100 - 100 / (1 + rs)

    # --- Synthetic Black–Scholes Delta of an ATM call whose "moneyness" is
    #     driven by the trailing z-score momentum. Smooth, bounded directional
    #     signal in (0, 1); 0.5 = neutral. ---
    roll_z = df["z_score"].rolling(10).mean().fillna(0.0)
    sig = df["vol_21"].fillna(df["vol_21"].median()).clip(0.05, 1.0)
    S_syn = 100.0 * np.exp(0.02 * roll_z)          # momentum tilts the spot
    df["bs_delta"] = bs_greeks(S_syn.values, 100.0, 0.25, 0.02, sig.values, "call")["delta"]
    df["bs_gamma"] = bs_greeks(S_syn.values, 100.0, 0.25, 0.02, sig.values, "call")["gamma"]

    return df

feat = build_features(prices)
FEATURE_COLS = [c for c in feat.columns if c not in ("close", "log_ret")]
print("Features:", FEATURE_COLS)
feat[FEATURE_COLS].tail()


In [ ]:
# --- Labels: next-day return classified with a volatility-scaled dead-band ---
next_ret = feat["log_ret"].shift(-1)                    # tomorrow's return
band = 0.25 * feat["log_ret"].rolling(21).std()         # tau ~ 1/4 of daily vol

label = pd.Series(0, index=feat.index)                  # 0 = Flat
label[next_ret > band] = 1                              # 1 = Long
label[next_ret < -band] = -1                            # -1 = Short

feat["label"] = label
feat["next_ret"] = next_ret

data = feat.dropna().copy()
data = data.iloc[:-1]        # drop last row (unknown future return)
print("Usable samples:", len(data))
print("Class balance (share):")
print((data["label"].value_counts(normalize=True)
       .rename({1: "Long", 0: "Flat", -1: "Short"}).round(3)))


<a id="train"></a>
## 8. Part II — Building & training the network

We use a compact **feed-forward network** (multilayer perceptron). Labels
$\{-1,0,+1\}$ are mapped to $\{0,1,2\}$ for a 3-way softmax, and the split is
**strictly chronological**.

### Controlling overfitting

Financial data has a **very low signal-to-noise ratio**, so a flexible network
will happily *memorise* the training set — its training accuracy climbs while
validation accuracy stays flat. That gap is overfitting, and it makes the
backtest meaningless. We fight it with several complementary regularizers:

| Technique | Why it helps |
|-----------|--------------|
| **Small network** (32 → 16 units) | fewer parameters ⇒ less capacity to memorise noise |
| **L2 weight decay** ($\lambda=3\times10^{-4}$) | penalises large weights, keeps the function smooth |
| **Dropout 0.45 + BatchNorm** | forces redundant, robust representations |
| **Label smoothing (0.05)** | stops the net from becoming over-confident on noisy labels |
| **Smaller LR + larger batch** | smoother, less noisy gradient steps |
| **Early stopping (patience 12)** | keeps the best *validation* weights, not the over-trained ones |

After training we print the **train-vs-validation accuracy gap** as a quick
overfitting diagnostic: a gap above roughly **0.05–0.08** means you should
regularize harder (raise dropout/L2, shrink the network, or add more data).


In [ ]:
# --- Chronological split: 70% train, 15% validation, 15% test ---
n = len(data)
i_tr, i_val = int(0.70 * n), int(0.85 * n)

X = data[FEATURE_COLS].values.astype("float32")
y = (data["label"].values + 1).astype("int64")   # {-1,0,1} -> {0,1,2}

X_tr,  y_tr  = X[:i_tr],       y[:i_tr]
X_val, y_val = X[i_tr:i_val],  y[i_tr:i_val]
X_te,  y_te  = X[i_val:],      y[i_val:]

# --- Standardize using TRAIN statistics only (no look-ahead) ---
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_val_s, X_te_s = map(scaler.transform, (X_tr, X_val, X_te))

# --- Class weights: markets are imbalanced (lots of Flat/small moves) ---
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_tr)
cw = compute_class_weight("balanced", classes=classes, y=y_tr)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

print(f"Train {len(X_tr)} | Val {len(X_val)} | Test {len(X_te)}")
print("Class weights:", {(-1,0,1)[k]: round(v,2) for k,v in class_weight.items()})


In [ ]:
from tensorflow.keras import regularizers

def build_model(n_features, n_classes=3, l2=3e-4, dropout=0.45):
    '''Small, heavily-regularized MLP.

    Anti-overfitting choices (see the markdown above):
    - low capacity (32 -> 16 units): fewer parameters than samples,
    - L2 weight decay on every dense layer,
    - high dropout + BatchNorm between layers,
    - label smoothing in the loss (built in the compile step).
    '''
    reg = regularizers.l2(l2)
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(32, activation="relu", kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax"),
    ])
    # Label smoothing needs one-hot targets, so use CategoricalCrossentropy.
    model.compile(
        optimizer=keras.optimizers.Adam(5e-4),   # smaller LR = smoother fit
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return model

model = build_model(X_tr_s.shape[1])
model.summary()

# One-hot targets (required by CategoricalCrossentropy + label smoothing)
n_classes = 3
y_tr_oh  = keras.utils.to_categorical(y_tr,  n_classes)
y_val_oh = keras.utils.to_categorical(y_val, n_classes)


In [ ]:
callbacks = [
    # Stop on the metric we actually care about, with a tight patience so the
    # model cannot keep memorising the training set after val stops improving.
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=6, min_lr=1e-5),
]

history = model.fit(
    X_tr_s, y_tr_oh,
    validation_data=(X_val_s, y_val_oh),
    epochs=200, batch_size=128,          # larger batch = less noisy, less overfit
    class_weight=class_weight,
    callbacks=callbacks, verbose=0,
)
print("Training finished. Epochs run:", len(history.history["loss"]))

# Quick overfitting diagnostic: gap between train and validation accuracy.
gap = history.history["accuracy"][-1] - history.history["val_accuracy"][-1]
print(f"Final train acc {history.history['accuracy'][-1]:.3f} | "
      f"val acc {history.history['val_accuracy'][-1]:.3f} | gap {gap:+.3f}")
print("Rule of thumb: a gap > ~0.05-0.08 signals overfitting.")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

proba_te = model.predict(X_te_s, verbose=0)
pred_te = proba_te.argmax(axis=1)

print("Classification report on the TEST set:")
print(classification_report(y_te, pred_te,
      target_names=["Short", "Flat", "Long"], zero_division=0))

cm = confusion_matrix(y_te, pred_te, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(5.2, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(["Short", "Flat", "Long"]); ax.set_yticklabels(["Short", "Flat", "Long"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (test)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()


<a id="backtest"></a>
## 9. Part III — Backtesting the strategy

### From probabilities to a position

Rather than a hard long/flat/short label, we convert the softmax probabilities
into a **continuous target position** in $[-1, +1]$ — a Delta-like exposure:

$$
\text{position}_t = \big(p^{\text{long}}_t - p^{\text{short}}_t\big)\cdot g,
$$

optionally scaled by a **volatility target** so we take *less* risk when the
market is turbulent (the BS $\sigma$ entering position sizing). We then apply the
position to **tomorrow's** return and subtract **transaction costs** proportional
to how much we trade.


In [ ]:
def backtest(dates, next_rets, proba, realized_vol,
             gain=1.0, cost_bps=1.0, vol_target=0.15, max_leverage=1.0):
    '''Vectorised long/flat/short backtest with costs and vol targeting.

    proba columns: 0=Short, 1=Flat, 2=Long.
    next_rets: the (already forward-aligned) next-day log returns for each row.
    '''
    p_short, p_long = proba[:, 0], proba[:, 2]
    raw_pos = np.clip((p_long - p_short) * gain, -1, 1)

    # Volatility targeting: scale exposure toward a target annualized vol
    rv = np.where(realized_vol > 1e-6, realized_vol, np.nan)
    scale = np.clip(vol_target / rv, 0, max_leverage)
    scale = pd.Series(scale).ffill().fillna(1.0).values
    position = np.clip(raw_pos * scale, -max_leverage, max_leverage)

    # Costs proportional to turnover
    turnover = np.abs(np.diff(position, prepend=0.0))
    costs = turnover * (cost_bps / 1e4)

    strat_ret = position * next_rets - costs
    bh_ret = next_rets  # buy & hold benchmark

    res = pd.DataFrame({
        "position": position,
        "strategy": strat_ret,
        "buy_hold": bh_ret,
    }, index=dates)
    res["equity_strategy"] = np.exp(res["strategy"].cumsum())
    res["equity_buyhold"] = np.exp(res["buy_hold"].cumsum())
    return res


test_dates = data.index[i_val:]
test_next_ret = data["next_ret"].values[i_val:]
test_vol = data["vol_21"].values[i_val:]

bt = backtest(test_dates, test_next_ret, proba_te, test_vol,
              gain=1.5, cost_bps=1.0, vol_target=0.15, max_leverage=1.0)
bt.tail()


In [ ]:
def performance_stats(returns, periods=252):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return {}
    cum = np.exp(r.sum()) - 1
    ann_ret = np.exp(r.mean() * periods) - 1
    ann_vol = r.std() * np.sqrt(periods)
    sharpe = (r.mean() / (r.std() + 1e-12)) * np.sqrt(periods)
    downside = r[r < 0].std() * np.sqrt(periods)
    sortino = (r.mean() * periods) / (downside + 1e-12)
    equity = np.exp(r.cumsum())
    dd = equity / equity.cummax() - 1
    max_dd = dd.min()
    calmar = ann_ret / (abs(max_dd) + 1e-12)
    win = (r > 0).mean()
    return {
        "Total return": f"{cum:6.1%}",
        "Annual return": f"{ann_ret:6.1%}",
        "Annual vol": f"{ann_vol:6.1%}",
        "Sharpe": f"{sharpe:6.2f}",
        "Sortino": f"{sortino:6.2f}",
        "Max drawdown": f"{max_dd:6.1%}",
        "Calmar": f"{calmar:6.2f}",
        "Win rate": f"{win:6.1%}",
    }

stats_strat = performance_stats(bt["strategy"])
stats_bh = performance_stats(bt["buy_hold"])
summary = pd.DataFrame({"NN strategy": stats_strat, "Buy & Hold": stats_bh})
print("Out-of-sample (test) performance:\n")
print(summary.to_string())


<a id="results"></a>
## 10. Part III — Results & charts

The panels below summarise the out-of-sample behaviour of the algorithm:
equity curve vs buy-&-hold, drawdown, the position the network takes over time,
and the distribution of daily P&L.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (1) Equity curves
ax = axes[0, 0]
ax.plot(bt.index, bt["equity_strategy"], lw=1.8, label="NN strategy")
ax.plot(bt.index, bt["equity_buyhold"], lw=1.4, label="Buy & Hold", alpha=0.8)
ax.set_title("Out-of-sample equity curve (growth of 1)")
ax.set_ylabel("Equity"); ax.legend()

# (2) Drawdown of the strategy
ax = axes[0, 1]
eq = bt["equity_strategy"]
dd = eq / eq.cummax() - 1
ax.fill_between(bt.index, dd, 0, color="crimson", alpha=0.4)
ax.set_title("Strategy drawdown"); ax.set_ylabel("Drawdown")

# (3) Position over time
ax = axes[1, 0]
ax.plot(bt.index, bt["position"], lw=1.0, color="teal")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("Network position (−1 short … +1 long)"); ax.set_ylabel("Exposure")

# (4) Distribution of daily strategy returns
ax = axes[1, 1]
ax.hist(bt["strategy"], bins=40, alpha=0.7, color="slateblue")
ax.axvline(bt["strategy"].mean(), color="black", ls="--",
           label=f"mean = {bt['strategy'].mean():.4f}")
ax.set_title("Distribution of daily strategy returns")
ax.set_xlabel("Daily log-return"); ax.legend()

plt.tight_layout(); plt.show()


In [ ]:
# Rolling 63-day (≈3 months) annualized Sharpe ratio, to see stability
roll = bt["strategy"].rolling(63)
roll_sharpe = (roll.mean() / (roll.std() + 1e-12)) * np.sqrt(252)

plt.figure(figsize=(12, 4))
plt.plot(bt.index, roll_sharpe, lw=1.4)
plt.axhline(0, color="gray", lw=0.8)
plt.axhline(1, color="green", ls="--", lw=0.8, label="Sharpe = 1")
plt.title("Rolling 63-day annualized Sharpe (strategy)")
plt.ylabel("Sharpe"); plt.legend(); plt.tight_layout(); plt.show()


<a id="conclusions"></a>
## 11. Conclusions & caveats

**What we built.**
- A **Black–Scholes simulation** engine (GBM paths, closed-form pricer, Greeks)
  that recovers the theoretical log-normal price / normal log-return behaviour.
- A **feature set grounded in the BS / volatility framework** (realized $\sigma$,
  standardized moves, a synthetic Delta/Gamma) feeding a **neural-network
  classifier** that outputs long / flat / short decisions.
- A **realistic backtest** on real historical data with volatility targeting and
  transaction costs, reported through a full set of **charts and metrics**.

**Honest caveats.**
- Daily direction is close to a **coin flip**; edges are thin and easily eaten by
  costs. Treat any positive Sharpe here as *illustrative*, not deployable.
- A single train/val/test split is optimistic. Production research uses
  **walk-forward / purged cross-validation** and multiple assets.
- No slippage, borrow costs, market impact, or regime-shift handling are modelled
  beyond a flat cost in bps.

**Natural extensions.**
- Swap the MLP for an **LSTM / Temporal CNN / Transformer** on return sequences.
- Predict **volatility** (a true BS input) and trade **options** with the Greeks
  computed above, rather than only the underlying.
- Use **implied volatility** and the **volatility surface** as extra features.
- Optimize the position map with **reinforcement learning** (reward = risk-adjusted
  P&L) instead of a fixed probability-to-Delta rule.

*Educational use only — not investment advice.*
